## Long Document Understanding with Qwen3-VL

In this notebook, we delve into the capabilities of the Qwen3-VL model for understanding **long document with hundreds of pages**. Our objective is to showcase how this advanced model can be applied to long/full-PDF document analysis scenarios.


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Required: OPENAI_API_KEY, OPENAI_BASE_URL, MODEL_ID
load_dotenv()

# Set OpenAI API credentials (required for OpenAI-compatible server)
API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_ID = os.environ["MODEL_ID"]

print("✓ Environment variables loaded successfully")
print(f"  API_KEY: {API_KEY[:8]}..." if API_KEY else "  API_KEY: (empty)")
print(f"  BASE_URL: {BASE_URL}")
print(f"  MODEL_ID: {MODEL_ID}")


#### \[Setup\]

We start by loading the pre-trained model.


In [ ]:
%pip install git+https://github.com/huggingface/transformers
%pip install qwen_vl_utils -U

Load PDF pages as images

In [ ]:
%pip install pdf2image

In [ ]:
import os
import math
import hashlib
import requests

from IPython.display import Markdown, display
import numpy as np
from PIL import Image
from pdf2image import convert_from_path


def download_file(url, dest_path):
    response = requests.get(url, stream=True)
    with open(dest_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8096):
            f.write(chunk)
    print(f"File downloaded to {dest_path}")


def get_pdf_images(pdf_path, dpi=144, cache_dir='cache'):
    os.makedirs(cache_dir, exist_ok=True)

    # Create a hash for the PDF path to use in cache filenames
    pdf_hash = hashlib.md5(pdf_path.encode('utf-8')).hexdigest()
    
    # Handle URL
    if pdf_path.startswith('http://') or pdf_path.startswith('https://'):
        pdf_file_path = os.path.join(cache_dir, f'{pdf_hash}.pdf')
        if not os.path.exists(pdf_file_path):
            download_file(pdf_path, pdf_file_path)
        else:
            print(f"Load cached PDF file from {pdf_file_path}.")
    else:
        pdf_file_path = pdf_path

    # Check for cached images
    images_cache_file = os.path.join(cache_dir, f'{pdf_hash}_{dpi}_images.npy')
    if os.path.exists(images_cache_file):
        images = np.load(images_cache_file, allow_pickle=True)
        pil_images = [Image.fromarray(image) for image in images]
        print(f"Load {len(images)} pages from cache: {images_cache_file}.")
        return pdf_file_path, pil_images

    # Convert PDF to images if not cached
    print(f"Converting PDF to images at {dpi} DPI...")
    pil_images = convert_from_path(pdf_file_path, dpi=dpi)
    
    # image file size control
    resize_pil_images = []
    for img in pil_images:
        width, height = img.size
        max_side = max(width, height)
        max_side_value = 1500
        if max_side > max_side_value:
            img = img.resize((width * max_side_value // max_side, height * max_side_value // max_side))
        resize_pil_images.append(img)
    pil_images = resize_pil_images
    
    images = [np.array(img) for img in pil_images]
    
    # Save to cache
    np.save(images_cache_file, images)
    print(f"Converted and cached {len(images)} pages to {images_cache_file}.")
    
    return pdf_file_path, pil_images


def create_image_grid(pil_images, num_columns=8):

    num_rows = math.ceil(len(pil_images) / num_columns)

    img_width, img_height = pil_images[0].size
    grid_width = num_columns * img_width
    grid_height = num_rows * img_height
    grid_image = Image.new('RGB', (grid_width, grid_height))

    for idx, image in enumerate(pil_images):
        row_idx = idx // num_columns
        col_idx = idx % num_columns
        position = (col_idx * img_width, row_idx * img_height)
        grid_image.paste(image, position)

    return grid_image

import base64
from io import BytesIO

def image_to_base64(img, format="PNG"):

    buffered = BytesIO()
    img.save(buffered, format=format)
    img_bytes = buffered.getvalue()
    img_base64 = base64.b64encode(img_bytes).decode('utf-8')
    
    return img_base64


Inference function with API using OpenAI SDK.

**Important Notice:**
- The API must support multi-image inputs.
- Ensure your API key has the necessary permissions and funds.
- For very long documents, be mindful of API limits on the number of total tokens or request size.

In [ ]:
from openai import OpenAI

def inference_with_api(images, prompt, sys_prompt="", model_id=MODEL_ID, min_pixels=590*32*32, max_pixels=730*32*32):
    client = OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL
    )
    print("Send {} pages to the model... \nWaiting for response...".format(len(images)))

    content_list = []
    for image in images:
        base64_image = image_to_base64(image)
        content_list.append(
            {
                "type": "image_url",
                # Pass in BASE64 image data. Note that the image format (i.e., image/{format}) must match the Content Type in the list of supported images. "f" is the method for string formatting.
                # PNG image:  f"data:image/png;base64,{base64_image}"
                # JPEG image: f"data:image/jpeg;base64,{base64_image}"
                # WEBP image: f"data:image/webp;base64,{base64_image}"
                "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                "min_pixels": min_pixels,
                "max_pixels": max_pixels,
            },
        )
    content_list.append({"type": "text", "text": prompt})
    messages = [
        # {
        #     "role": "system",
        #     "content": [{"type":"text","text": sys_prompt}]
        # },
        {
            "role": "user",
            "content": content_list
        }
    ]

    completion = client.chat.completions.create(
        model=model_id,
        messages=messages,
        # top_p=0.8,
        # temperature=0.01,
        # presence_penalty=1.5,
        # max_tokens=16384,
        # extra_body={
        #     'top_k': 1,
        #     'repetition_penalty': 1.0,
        # },
    )
    return completion.choices[0].message.content

### Example: Analyzing an Academic PDF

In this section, we demonstrate how the model can be used to read and understand a long PDF document. We will use an academic file and ask the model to analyze it.

#### 1. Use API for inference.

In [ ]:
# Using a PDF document URL or local path.
# longdoc_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-VL/demo/longdoc/documents/fox_got_merge_code.pdf"
# prompt = "How many tables?"


# Using a PDF document URL or local path. 
longdoc_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-VL/demo/longdoc/documents/Qwen2.5-VL.pdf"
prompt = "Please summarize the key contributions of this paper based on its abstract and introduction."


# This will download the PDF, convert its pages to images, and then run inference with API.
pdf_path, images = get_pdf_images(longdoc_url, dpi=144)

# You can use this to visualize documents in thumbnail format.
image_grid = create_image_grid(images, num_columns=8)
display(image_grid.resize((1500, 1100)))
print(images[0].size)

response = inference_with_api(images, prompt)
display(Markdown(response))